# Notes

# Setup

In [2]:
print('hello')

hello


In [3]:
%load_ext autoreload
%autoreload 2

import os
import re
from scipy import stats

import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
import numpy as np
import rasterio

import config
import hatfield_funcs as hf
import hatfield_funcs.helpers as hfh

from pathlib import Path
import sys

project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root / "src"))

from my_extra_to_hatfield_funcs.io_funcs import (
    read_xlsx,
    read_df_from_csv,
    write_df_to_csv,
    read_geojson,
    write_geojson,
)

from biofilm_data.io import (
    load_full_station_dataset,
    load_calibration_curves,
    load_lab_data,
)

from biofilm_data.plot import (
    plot_r_vs_s_comparison,
    plot_area_round_boxplots,
    internal_correlation_heatmap,
    external_correlation_heatmap,
)

from biofilm_data.calibration_curve import (
    fit_linear_calibration,
    plot_calibration_fit,
    mean_response_with_ci,
)


plt.style.use("default")
%config InlineBackend.figure_format = 'retina'

In [4]:
if os.getenv("GA_STORAGE_ACCOUNT_NAME") is None:
    raise RuntimeError(
        "This notebook is configured for Azure cloud storage. "
        "Set GA_STORAGE_ACCOUNT_NAME before running it."
    )

hf.config.configure_azure()
hf.config.configure_data_root_path(config.CONTAINER_NAME)

# Parameters

# Read Data

In [5]:
df_lab = load_lab_data(config.CONTAINER_FILE_PATHS["lab_data_csv"])
df_lab_R = df_lab.xs('R', level='sample_type')
display(df_lab_R.head())

Sample Total Dry Mass (g)  \
site      round station_id                              
brunswick 1     B1                         124.920787   
                B10                        113.372704   
                B11                         68.488373   
                B12                        209.965738   
                B13                        113.503925   

                            Extraction Dry Mass (mg)  Peri (mg/g)  \
site      round station_id                                          
brunswick 1     B1                             554.2     0.000130   
                B10                            460.8     0.000159   
                B11                            480.4     0.000083   
                B12                            484.9     0.000055   
                B13                            489.1     0.000034   

                            Chlc2 (mg/g)  Fuco (mg/g)  Ddx (mg/g)  \
site      round station_id                                          
brunswick 1     B1              0.000126     0.004335    0.003132   
                B10             0.000414     0.005671    0.003579   
                B11             0.000451     0.006007    0.004110   
                B12             0.000186     0.003457    0.001852   
                B13             0.000279     0.003589    0.001709   

                            Allo (mg/g)  Myxo (mg/g)  Lut (mg/g)  Zea (mg/g)  \
site      round station_id                                                     
brunswick 1     B1             0.000217     0.000186    0.000786    0.000417   
                B10            0.000381     0.000401    0.001142    0.000503   
                B11            0.000346     0.000399    0.001323    0.000442   
                B12            0.000214     0.000225    0.000480    0.000282   
                B13            0.000126     0.000159    0.000315    0.000114   

                            ...  Chla + allomers + DVCh-a : Chla + allomers  \
site      round station_id  ...                                               
brunswick 1     B1          ...                                    1.051073   
                B10         ...                                    1.104097   
                B11         ...                                    1.000000   
                B12         ...                                    1.077257   
                B13         ...                                    1.064407   

                            Chla + allomers + DVCh-a + Phta : Chla + allomers  \
site      round station_id                                                      
brunswick 1     B1                                                   1.116023   
                B10                                                  1.198461   
                B11                                                  1.052384   
                B12                                                  1.125541   
                B13                                                  1.134163   

                            Total FA : Chla + allomers  \
site      round station_id                               
brunswick 1     B1                           31.460557   
                B10                          24.753145   
                B11                          17.531168   
                B12                          11.290122   
                B13                          12.812582   

                            Total SFA : Chla + allomers  \
site      round station_id                                
brunswick 1     B1                            14.880862   
                B10                           13.840168   
                B11                            9.548230   
                B12                            5.052700   
                B13                            4.951975   

                            Total MUFA : Chla + allomers  \
site      round station_id                                 
brunswick 1     B1                  

In [6]:
# df_all_R = load_full_station_dataset(
#     config.CONTAINER_FILE_PATHS["full_station_dataset_R_geojson"],
#     r_only=True
# )

# display(df_all_R.head())

# # df_all = load_full_station_dataset(
# #     config.CONTAINER_FILE_PATHS["full_station_dataset_geojson"], r_only=False
# # )

# # display(df_all.head())

# Functions to externatlize

In [7]:
from typing import Any, Mapping, Sequence
from scipy.stats import energy_distance, ks_2samp, t, ttest_1samp, ttest_ind

def welch_mean_test(x_ref: np.ndarray, y_other: np.ndarray, alpha: float) -> dict[str, Any]:
    nx, ny = len(x_ref), len(y_other)
    estimate = float(np.mean(y_other) - np.mean(x_ref))
    var_x = float(np.var(x_ref, ddof=1))
    var_y = float(np.var(y_other, ddof=1))
    variance_estimate = var_x / nx + var_y / ny
    standard_error = float(np.sqrt(variance_estimate))
    result = ttest_ind(x_ref, y_other, equal_var=False, alternative="two-sided")

    denominator = (var_x / nx) ** 2 / (nx - 1) + (var_y / ny) ** 2 / (ny - 1)
    degrees_of_freedom = variance_estimate**2 / denominator if denominator else np.nan
    if standard_error and np.isfinite(degrees_of_freedom):
        critical = t.ppf(1 - alpha / 2, degrees_of_freedom)
        confidence_interval = (
            estimate - float(critical) * standard_error,
            estimate + float(critical) * standard_error,
        )
    else:
        confidence_interval = (estimate, estimate)

    pooled_sd = np.sqrt(((nx - 1) * var_x + (ny - 1) * var_y) / (nx + ny - 2))
    cohens_d = estimate / pooled_sd if pooled_sd > 0 else np.nan
    correction_denominator = 4 * (nx + ny) - 9
    correction = 1 - 3 / correction_denominator if correction_denominator else np.nan
    hedges_g = cohens_d * correction

    return {
        "method": "welch_t",
        "estimate": estimate,
        "standard_error": standard_error,
        "confidence_level": float(1 - alpha),
        "confidence_interval": tuple(map(float, confidence_interval)),
        "statistic": float(result.statistic),
        "degrees_of_freedom": float(degrees_of_freedom),
        "p_value": float(result.pvalue),
        "effect_size": float(hedges_g),
        "effect_size_name": "Hedges g",
    }


def paired_mean_test(differences: np.ndarray, alpha: float) -> dict[str, Any]:
    n = len(differences)
    estimate = float(np.mean(differences))
    standard_deviation = float(np.std(differences, ddof=1))
    standard_error = standard_deviation / np.sqrt(n)
    result = ttest_1samp(differences, popmean=0, alternative="two-sided")
    degrees_of_freedom = n - 1

    if standard_error:
        critical = t.ppf(1 - alpha / 2, degrees_of_freedom)
        confidence_interval = (
            estimate - float(critical) * standard_error,
            estimate + float(critical) * standard_error,
        )
    else:
        confidence_interval = (estimate, estimate)

    return {
        "method": "paired_t",
        "estimate": estimate,
        "standard_error": float(standard_error),
        "confidence_level": float(1 - alpha),
        "confidence_interval": tuple(map(float, confidence_interval)),
        "statistic": float(result.statistic),
        "degrees_of_freedom": int(degrees_of_freedom),
        "p_value": float(result.pvalue),
        "effect_size": float(estimate / standard_deviation)
        if standard_deviation > 0
        else np.nan,
        "effect_size_name": "Cohen dz",
    }


def holm_adjust(p_values: Sequence[float]) -> np.ndarray:
    """Adjust finite p-values while preserving missing values."""
    values = np.asarray(p_values, dtype=float)
    adjusted = np.full(values.shape, np.nan, dtype=float)
    valid = np.isfinite(values)
    if not valid.any():
        return adjusted

    finite_values = values[valid]
    order = np.argsort(finite_values)
    ranked = finite_values[order]
    corrected = np.maximum.accumulate(
        ranked * (len(ranked) - np.arange(len(ranked)))
    )
    corrected = np.minimum(corrected, 1.0)
    restored = np.empty_like(corrected)
    restored[order] = corrected
    adjusted[valid] = restored
    return adjusted


def energy_permutation_test(
    x_ref: np.ndarray,
    y_other: np.ndarray,
    *,
    n_permutations: int,
    seed: int | None,
    paired: bool,
) -> dict[str, Any]:
    nx, ny = len(x_ref), len(y_other)
    observed_distance = float(energy_distance(x_ref, y_other))
    scale = nx * ny / (nx + ny)
    observed_statistic = float(scale * observed_distance**2)
    rng = np.random.default_rng(seed)
    exceedances = 0

    if paired:
        for _ in range(n_permutations):
            swap = rng.integers(0, 2, size=nx, dtype=np.int8).astype(bool)
            permuted_x = np.where(swap, y_other, x_ref)
            permuted_y = np.where(swap, x_ref, y_other)
            distance = energy_distance(permuted_x, permuted_y)
            exceedances += scale * distance**2 >= observed_statistic
        method = "paired_energy_permutation"
    else:
        pooled = np.concatenate([x_ref, y_other])
        for _ in range(n_permutations):
            shuffled = rng.permutation(pooled)
            distance = energy_distance(shuffled[:nx], shuffled[nx:])
            exceedances += scale * distance**2 >= observed_statistic
        method = "energy_permutation"

    p_value = (exceedances + 1) / (n_permutations + 1)
    return {
        "method": method,
        "energy_distance": observed_distance,
        "statistic": observed_statistic,
        "p_value": float(p_value),
        "n_permutations": int(n_permutations),
    }

def compare_two_samples(
    sample_a_ref: Sequence[float] | pd.Series,
    sample_b_other: Sequence[float] | pd.Series,
    *,
    variable: str = "Variable",
    paired: bool = False,
    alpha: float = 0.05,
    n_permutations: int = 9999,
    seed: int | None = 42,
    run_energy_test: bool = True,
) -> dict[str, Any]:
    """Compare two samples, using matched tests only when ``paired=True``.

    Independent samples receive a Welch mean test, an energy-distance
    permutation test for a full-distribution difference, and a two-sample KS
    test. Paired samples are aligned by Series index (or position for arrays);
    their mean test is paired t and the energy test permutes labels within each
    pair, assuming the two labels are exchangeable within pairs under the null.
    The paired KS distance is descriptive only because observations are
    dependent.
    """
    def _numeric_values(values: Any, label: str) -> np.ndarray:
        series = pd.Series(values, copy=False)
        numeric = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
        result = numeric.dropna().to_numpy(dtype=float)
        if result.size < 2:
            raise ValueError(f"{label} must contain at least two finite observations.")
        return result

    def _sample_summary(values: np.ndarray) -> dict[str, float | int]:
        q1, q3 = np.quantile(values, [0.25, 0.75])
        return {
            "n": int(values.size),
            "mean": float(np.mean(values)),
            "sd": float(np.std(values, ddof=1)),
            "median": float(np.median(values)),
            "q1": float(q1),
            "q3": float(q3),
            "min": float(np.min(values)),
            "max": float(np.max(values)),
        }

    if not 0 < alpha < 1:
        raise ValueError("alpha must be between zero and one.")
    if run_energy_test and (int(n_permutations) != n_permutations or n_permutations < 1):
        raise ValueError("n_permutations must be a positive integer.")

    label_a = "Sample A"
    label_b = "Sample B"

    unmatched_a = unmatched_b = 0
    if paired:
        if isinstance(sample_a_ref, pd.Series) and isinstance(sample_b_other, pd.Series):
            if not sample_a_ref.index.is_unique or not sample_b_other.index.is_unique:
                raise ValueError("Paired Series indices must uniquely identify subjects.")
            aligned = pd.concat(
                [sample_a_ref.rename("a"), sample_b_other.rename("b")],
                axis=1,
                join="inner",
            )
            unmatched_a = len(sample_a_ref.index.difference(sample_b_other.index))
            unmatched_b = len(sample_b_other.index.difference(sample_a_ref.index))
            aligned = aligned.apply(pd.to_numeric, errors="coerce").replace(
                [np.inf, -np.inf], np.nan
            ).dropna()
            x = aligned["a"].to_numpy(dtype=float)
            y = aligned["b"].to_numpy(dtype=float)
        else:
            raw_a = pd.to_numeric(pd.Series(sample_a_ref), errors="coerce").to_numpy(dtype=float)
            raw_b = pd.to_numeric(pd.Series(sample_b_other), errors="coerce").to_numpy(dtype=float)
            if len(raw_a) != len(raw_b):
                raise ValueError("Paired array samples must have equal lengths.")
            valid = np.isfinite(raw_a) & np.isfinite(raw_b)
            unmatched_a = unmatched_b = int((~valid).sum())
            x, y = raw_a[valid], raw_b[valid]
        if len(x) < 2:
            raise ValueError("At least two complete matched pairs are required.")
        mean_test = paired_mean_test(y - x, alpha)
    else:
        x = _numeric_values(sample_a_ref, label_a)
        y = _numeric_values(sample_b_other, label_b)
        mean_test = welch_mean_test(x, y, alpha)

    if len(x) < 2 or len(y) < 2:
        raise ValueError("Each sample must contain at least two finite observations.")

    ks = ks_2samp(x, y, alternative="two-sided", method="auto")
    if run_energy_test:
        distribution_test = energy_permutation_test(
            x,
            y,
            n_permutations=int(n_permutations),
            seed=seed,
            paired=paired,
        )
        distribution_test["significant"] = bool(distribution_test["p_value"] < alpha)
    else:
        distribution_test = {
            "method": "not_run",
            "energy_distance": np.nan,
            "statistic": np.nan,
            "p_value": np.nan,
            "n_permutations": 0,
            "significant": False,
        }

    if paired:
        ks_result = {"distance": float(ks.statistic), "p_value": np.nan}
    else:
        ks_result = {"distance": float(ks.statistic), "p_value": float(ks.pvalue)}

    return {
        "variable": variable,
        "design": "paired" if paired else "independent",
        "alpha": float(alpha),
        "sample_a": _sample_summary(x),
        "sample_b": _sample_summary(y),
        "n_pairs": int(len(x)) if paired else 0,
        "unmatched_a": int(unmatched_a),
        "unmatched_b": int(unmatched_b),
        "mean_test": mean_test,
        "distribution_test": distribution_test,
        "ks_test": ks_result,
    }

# Functions Here

In [30]:
def perform_tests_between_sites(df, vars):
    results = []
    for var in vars:
        if 'round' in df.index.names:
            for round_ in config.ROUNDS:
                res = compare_two_samples(
                    df.xs(("westham", round_), level=("site", "round"))[var],
                    df.xs(("brunswick", round_), level=("site", "round"))[var],
                    paired=False,
                )
                result = {
                    "round": round_,
                    "variable": var,
                    "mean_difference": res["mean_test"]["estimate"],
                    "mean_test_p": res["mean_test"]["p_value"],
                    "distribution_test_p": res["distribution_test"]["p_value"],
                }
                results.append(result)
        else:
            round_ = 'none'
            res = compare_two_samples(
                df.xs("westham", level="site")[var],
                df.xs("brunswick", level="site")[var],
                paired=False,
            )
            result = {
                "round": 'none',
                "variable": var,
                "mean_difference": res["mean_test"]["estimate"],
                "mean_test_p": res["mean_test"]["p_value"],
                "distribution_test_p": res["distribution_test"]["p_value"],
            }
            results.append(result)

    df_results = pd.DataFrame(results).set_index(["variable", "round"])

    df_results["mean_test_p_adj"] = holm_adjust(df_results["mean_test_p"])
    df_results["distribution_test_p_adj"] = holm_adjust(df_results["distribution_test_p"])

    def is_p_significant(p_value, alpha=0.05):
        return p_value < alpha


    df_results["mean_test_significant"] = df_results["mean_test_p"].apply(is_p_significant)
    df_results["distribution_test_significant"] = df_results["distribution_test_p"].apply(
        is_p_significant
    )
    df_results["mean_test_significant_adj"] = df_results["mean_test_p_adj"].apply(
        is_p_significant
    )
    df_results["distribution_test_significant_adj"] = df_results[
        "distribution_test_p_adj"
    ].apply(is_p_significant)
    return df_results

def print_test_results(df_results):
    print('Both tests significant:')
    display(
        df_results[df_results["mean_test_significant_adj"] & df_results["distribution_test_significant_adj"]]
    )
    print('Only distribution test significant:')
    display(df_results[~df_results["mean_test_significant_adj"] & df_results["distribution_test_significant_adj"]])
    print('Only mean test significant:')
    display(
        df_results[
            df_results["mean_test_significant_adj"]
            & ~df_results["distribution_test_significant_adj"]
        ]
    )
    print('Neither test significant:')
    display(
        df_results[
            ~df_results["mean_test_significant_adj"]
            & ~df_results["distribution_test_significant_adj"]
        ]
    )
    return None



def perform_tests_between_rounds(df, vars):
    results = []
    for var in vars:
        for site in config.SITES:
            res = compare_two_samples(
                df.xs((site, 1), level=("site", "round"))[var],
                df.xs((site, 2), level=("site", "round"))[var],
                paired=True,
            )
            result = {
                "site": site,
                "variable": var,
                "mean_difference": res["mean_test"]["estimate"],
                "mean_test_p": res["mean_test"]["p_value"],
                "distribution_test_p": res["distribution_test"]["p_value"],
            }
            results.append(result)

    df_results = pd.DataFrame(results).set_index(["variable", "site"])

    df_results["mean_test_p_adj"] = holm_adjust(df_results["mean_test_p"])
    df_results["distribution_test_p_adj"] = holm_adjust(df_results["distribution_test_p"])

    def is_p_significant(p_value, alpha=0.05):
        return p_value < alpha


    df_results["mean_test_significant"] = df_results["mean_test_p"].apply(is_p_significant)
    df_results["distribution_test_significant"] = df_results["distribution_test_p"].apply(
        is_p_significant
    )
    df_results["mean_test_significant_adj"] = df_results["mean_test_p_adj"].apply(
        is_p_significant
    )
    df_results["distribution_test_significant_adj"] = df_results[
        "distribution_test_p_adj"
    ].apply(is_p_significant)
    return df_results

def print_test_results(df_results):
    print('Both tests significant:')
    display(
        df_results[df_results["mean_test_significant_adj"] & df_results["distribution_test_significant_adj"]]
    )
    print('Only distribution test significant:')
    display(df_results[~df_results["mean_test_significant_adj"] & df_results["distribution_test_significant_adj"]])
    print('Only mean test significant:')
    display(
        df_results[
            df_results["mean_test_significant_adj"]
            & ~df_results["distribution_test_significant_adj"]
        ]
    )
    print('Neither test significant:')
    display(
        df_results[
            ~df_results["mean_test_significant_adj"]
            & ~df_results["distribution_test_significant_adj"]
        ]
    )
    return None



# Distribution tests

In [9]:
var_list_dict = {
    "Core Pigments": [
        "Chla + allomers (mg/m2)",
        "Fuco (mg/m2)",
        "Ddx (mg/m2)",
        "DVCh-a (mg/m2)",
        "Zea (mg/m2)",
        "Chlb (mg/m2)",
        "Phta (mg/m2)",
        "Chla + allomers : Phta",
    ],
    "Fatty Acids and Nutritional Components": [
        "Total SFA (mg/m2)",
        "Total MUFA (mg/m2)",
        "Total PUFA (mg/m2)",
        "Total FA (mg/m2)",
        "Total SFA : Total FA (%)",
        "Total MUFA : Total FA (%)",
        "Total PUFA : Total FA (%)",
        "Total Carbs (mg/m2)",
        "Caloric (kJ/m2)",
        "Proteins (mg/m2)",
    ],
    "Sediment Sample Components": [
        "Sample Total Dry Mass (g)",
        "Moisture (%)",
        "Percent Sand (%)",
        "Percent Silt (%)",
        "Percent Clay (%)",
    ],
}

## Between sites

In [10]:
for category, variables in var_list_dict.items():
    print("\n", "=" * 50, "\n")
    print(f"Testing category: {category}")
    print("\n", "-" * 50, "\n")
    df_results = perform_tests_between_sites(df_lab_R, variables)
    print_test_results(df_results)
    print("\n", '='*50, "\n")



Testing category: Core Pigments

 -------------------------------------------------- 

Both tests significant:


mean_difference   mean_test_p  \
variable                round                                  
Chla + allomers (mg/m2) 1             7.042591  3.661871e-03   
                        2            13.037932  2.022835e-07   
Fuco (mg/m2)            1             1.793395  2.621087e-03   
                        2             2.886628  1.686449e-08   
Ddx (mg/m2)             2             1.123159  3.575736e-04   
DVCh-a (mg/m2)          2             1.348389  6.160217e-05   
Zea (mg/m2)             1             0.336391  2.866623e-03   
                        2             0.277281  2.905383e-05   
Phta (mg/m2)            2             1.070620  2.299608e-06   

                               distribution_test_p  mean_test_p_adj  \
variable                round                                         
Chla + allomers (mg/m2) 1                   0.0016     2.929497e-02   
                        2                   0.0001     3.034253e-06   
Fuco (mg/m2)            1                   0.0004     2.621087e-02   
                        2                   0.0001     2.698319e-07   
Ddx (mg/m2)             2                   0.0008     3.933310e-03   
DVCh-a (mg/m2)          2                   0.0001     7.392260e-04   
Zea (mg/m2)             1                   0.0011     2.621087e-02   
                        2                   0.0001     3.776998e-04   
Phta (mg/m2)            2                   0.0001     3.219452e-05   

                               distribution_test_p_adj  mean_test_significant  \
variable                round                                                   
Chla + allomers (mg/m2) 1                       0.0128                   True   
                        2                       0.0016                   True   
Fuco (mg/m2)            1                       0.0044                   True   
                        2                       0.0016                   True   
Ddx (mg/m2)             2                       0.0080                   True   
DVCh-a (mg/m2)          2                       0.0016                   True   
Zea (mg/m2)             1                       0.0099                   True   
                        2                       0.0016                   True   
Phta (mg/m2)            2                       0.0016                   True   

                               distribution_test_significant  \
variable                round                                  
Chla + allomers (mg/m2) 1                               True   
                        2                               True   
Fuco (mg/m2)            1                               True   
                        2                               True   
Ddx (mg/m2)             2                               True   
DVCh-a (mg/m2)          2                               True   
Zea (mg/m2)             1                               True   
                        2                               True   
Phta (mg/m2)            2                               True   

                               mean_test_significant_adj  \
variable                round                              
Chla + allomers (mg/m2) 1                           True   
                        2                           True   
Fuco (mg/m2)            1                           True   
                        2                           True   
Ddx (mg/m2)             2                           True   
DVCh-a (mg/m2)          2                           True   
Zea (mg/m2)             1                           True   
                        2                           True   
Phta (mg/m2)            2                           True   

                               distribution_test_significant_adj  
variable                round                                     
Chla + allomers (mg/m2) 1                                   True  
                        2                                   True  
Fuco (mg/m2)      

Only distribution test significant:


mean_difference  mean_test_p  \
variable               round                                 
DVCh-a (mg/m2)         1             0.419646     0.021415   
Chlb (mg/m2)           1             0.198499     0.074645   
                       2             0.221308     0.036044   
Phta (mg/m2)           1             0.822055     0.193925   
Chla + allomers : Phta 1            -7.785382     0.012091   
                       2            -4.357110     0.032162   

                              distribution_test_p  mean_test_p_adj  \
variable               round                                         
DVCh-a (mg/m2)         1                   0.0043         0.128491   
Chlb (mg/m2)           1                   0.0060         0.223934   
                       2                   0.0134         0.160811   
Phta (mg/m2)           1                   0.0047         0.387851   
Chla + allomers : Phta 1                   0.0053         0.084640   
                       2                   0.0205         0.160811   

                              distribution_test_p_adj  mean_test_significant  \
variable               round                                                   
DVCh-a (mg/m2)         1                       0.0301                   True   
Chlb (mg/m2)           1                       0.0301                  False   
                       2                       0.0402                   True   
Phta (mg/m2)           1                       0.0301                  False   
Chla + allomers : Phta 1                       0.0301                   True   
                       2                       0.0410                   True   

                              distribution_test_significant  \
variable               round                                  
DVCh-a (mg/m2)         1                               True   
Chlb (mg/m2)           1                               True   
                       2                               True   
Phta (mg/m2)           1                               True   
Chla + allomers : Phta 1                               True   
                       2                               True   

                              mean_test_significant_adj  \
variable               round                              
DVCh-a (mg/m2)         1                          False   
Chlb (mg/m2)           1                          False   
                       2                          False   
Phta (mg/m2)           1                          False   
Chla + allomers : Phta 1                          False   
                       2                          False   

                              distribution_test_significant_adj  
variable               round                                     
DVCh-a (mg/m2)         1                                   True  
Chlb (mg/m2)           1                                   True  
                       2                                   True  
Phta (mg/m2)           1                                   True  
Chla + allomers : Phta 1                                   True  
                       2                                   True

Only mean test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,


... (14 outputs left out)

Only mean test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,


Neither test significant:


mean_difference  mean_test_p  \
variable                  round                                 
Sample Total Dry Mass (g) 1            -0.140045     0.992082   
Moisture (%)              1             0.109398     0.964903   
                          2            -6.649249     0.010069   
Percent Sand (%)          1            -4.476431     0.218339   
                          2             4.027499     0.050529   
Percent Silt (%)          1             4.317339     0.222966   
                          2            -3.348579     0.126640   
Percent Clay (%)          1             0.159092     0.675448   
                          2            -0.678921     0.132442   

                                 distribution_test_p  mean_test_p_adj  \
variable                  round                                         
Sample Total Dry Mass (g) 1                   0.0396         1.000000   
Moisture (%)              1                   0.5408         1.000000   
                          2                   0.0168         0.090623   
Percent Sand (%)          1                   0.5439         1.000000   
                          2                   0.0590         0.404232   
Percent Silt (%)          1                   0.5352         1.000000   
                          2                   0.1364         0.886481   
Percent Clay (%)          1                   0.7148         1.000000   
                          2                   0.2350         0.886481   

                                 distribution_test_p_adj  \
variable                  round                            
Sample Total Dry Mass (g) 1                       0.3168   
Moisture (%)              1                       1.0000   
                          2                       0.1512   
Percent Sand (%)          1                       1.0000   
                          2                       0.4130   
Percent Silt (%)          1                       1.0000   
                          2                       0.8184   
Percent Clay (%)          1                       1.0000   
                          2                       1.0000   

                                 mean_test_significant  \
variable                  round                          
Sample Total Dry Mass (g) 1                      False   
Moisture (%)              1                      False   
                          2                       True   
Percent Sand (%)          1                      False   
                          2                      False   
Percent Silt (%)          1                      False   
                          2                      False   
Percent Clay (%)          1                      False   
                          2                      False   

                                 distribution_test_significant  \
variable                  round                                  
Sample Total Dry Mass (g) 1                               True   
Moisture (%)              1                              False   
                          2                               True   
Percent Sand (%)          1                              False   
                          2                              False   
Percent Silt (%)          1                              False   
                          2                              False   
Percent Clay (%)          1                              False   
                          2                              False   

                                 mean_test_significant_adj  \
variable                  round                              
Sample Total Dry Mass (g) 1                          False   
Moisture (%)              1                          False   
                          2                          False   
Percent Sand (%)          1                          False   
                          2                          False   
Percent Silt (%)          1                         

## Between rounds

In [11]:
for category, variables in var_list_dict.items():
    print("\n", "=" * 50, "\n")
    print(f"Testing category: {category}")
    print("\n", "-" * 50, "\n")
    df_results = perform_tests_between_rounds(df_lab_R, variables)
    print_test_results(df_results)
    print("\n", '='*50, "\n")



Testing category: Core Pigments

 -------------------------------------------------- 

Both tests significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,site,,,,,,,,,
Fuco (mg/m2),westham,-1.784288,0.000334,0.0001,0.005337,0.0016,True,True,True,True


Only distribution test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,site,,,,,,,,,
Phta (mg/m2),westham,-0.844055,0.099516,0.0016,0.796124,0.024,False,True,False,True


Only mean test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,site,,,,,,,,,


... (14 outputs left out)

Only mean test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,site,,,,,,,,,


Neither test significant:


mean_difference  mean_test_p  \
variable                  site                                      
Sample Total Dry Mass (g) brunswick        12.180153     0.189305   
Moisture (%)              brunswick        -1.104709     0.366169   
                          westham           5.653938     0.015948   
Percent Sand (%)          brunswick         7.294289     0.011742   
                          westham          -1.209641     0.549053   
Percent Silt (%)          brunswick        -6.929037     0.011647   
                          westham           0.736880     0.735762   
Percent Clay (%)          brunswick        -0.365252     0.229751   
                          westham           0.472761     0.218066   

                                     distribution_test_p  mean_test_p_adj  \
variable                  site                                              
Sample Total Dry Mass (g) brunswick               0.0823         1.000000   
Moisture (%)              brunswick               0.3724         1.000000   
                          westham                 0.0180         0.111635   
Percent Sand (%)          brunswick               0.0098         0.104825   
                          westham                 0.5913         1.000000   
Percent Silt (%)          brunswick               0.0162         0.104825   
                          westham                 0.7908         1.000000   
Percent Clay (%)          brunswick               0.2525         1.000000   
                          westham                 0.3809         1.000000   

                                     distribution_test_p_adj  \
variable                  site                                 
Sample Total Dry Mass (g) brunswick                   0.4938   
Moisture (%)              brunswick                   1.0000   
                          westham                     0.1296   
Percent Sand (%)          brunswick                   0.0882   
                          westham                     1.0000   
Percent Silt (%)          brunswick                   0.1296   
                          westham                     1.0000   
Percent Clay (%)          brunswick                   1.0000   
                          westham                     1.0000   

                                     mean_test_significant  \
variable                  site                               
Sample Total Dry Mass (g) brunswick                  False   
Moisture (%)              brunswick                  False   
                          westham                     True   
Percent Sand (%)          brunswick                   True   
                          westham                    False   
Percent Silt (%)          brunswick                   True   
                          westham                    False   
Percent Clay (%)          brunswick                  False   
                          westham                    False   

                                     distribution_test_significant  \
variable                  site                                       
Sample Total Dry Mass (g) brunswick                          False   
Moisture (%)              brunswick                          False   
                          westham                             True   
Percent Sand (%)          brunswick                           True   
                          westham                            False   
Percent Silt (%)          brunswick                           True   
                          westham                            False   
Percent Clay (%)          brunswick                          False   
                          westham                            False   

                                     mean_test_significant_adj  \
variable                  site                                   
Sample Total Dry Mass (g) brunswick                      False   
Moisture (%)              brunswick                      False   
      

## Between sites for round absolute and relative differences

In [ ]:
columns_to_keep = [col for cols in var_list_dict.values() for col in cols]

df_lab_R_round_1 = df_lab_R.xs(1, level='round')[columns_to_keep]
df_lab_R_round_2 = df_lab_R.xs(2, level='round')[columns_to_keep]

df_lab_R_diff_abs = df_lab_R_round_2 - df_lab_R_round_1
df_lab_R_diff_rel_pc = 100 * (df_lab_R_diff_abs / df_lab_R_round_1)

# Replace non finite values by NaN
df_lab_R_diff_rel_pc = df_lab_R_diff_rel_pc.replace([np.inf, -np.inf], np.nan)

In [31]:
for category, variables in var_list_dict.items():
    print("\n", "=" * 50, "\n")
    print(f"Testing category: {category}")
    print("\n", "-" * 50, "\n")
    df_results = perform_tests_between_sites(df_lab_R_diff_abs, variables)
    print_test_results(df_results)
    print("\n", "=" * 50, "\n")




Testing category: Core Pigments

 -------------------------------------------------- 



Both tests significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,
DVCh-a (mg/m2),none,0.928743,0.0048,0.0016,0.038401,0.0128,True,True,True,True


Only distribution test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,


Only mean test significant:


... (15 outputs left out)

Only mean test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,
Percent Sand (%),none,8.50393,0.014557,0.0203,0.04367,0.0609,True,True,True,False


Neither test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,
Percent Silt (%),none,-7.665917,0.026371,0.0323,0.052741,0.0646,True,True,False,False
Percent Clay (%),none,-0.838013,0.085899,0.0740,0.085899,0.0740,False,False,False,False


In [32]:
for category, variables in var_list_dict.items():
    print("\n", "=" * 50, "\n")
    print(f"Testing category: {category}")
    print("\n", "-" * 50, "\n")
    df_results = perform_tests_between_sites(df_lab_R_diff_rel_pc, variables)
    print_test_results(df_results)
    print("\n", "=" * 50, "\n")




Testing category: Core Pigments

 -------------------------------------------------- 

Both tests significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,


Only distribution test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,


Only mean test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,


... (14 outputs left out)

Only mean test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,


Neither test significant:


,,mean_difference,mean_test_p,distribution_test_p,mean_test_p_adj,distribution_test_p_adj,mean_test_significant,distribution_test_significant,mean_test_significant_adj,distribution_test_significant_adj
variable,round,,,,,,,,,
Percent Sand (%),none,23.066059,0.056366,0.0210,0.144026,0.063,False,True,False,False
Percent Silt (%),none,-59.324747,0.081961,0.0327,0.144026,0.063,False,True,False,False
Percent Clay (%),none,-93.981360,0.048009,0.0257,0.144026,0.063,True,True,False,False
